# Trenovani modelu pro predikci fotbalovych zapasu

Tento notebook je hlavni a autoritativni workflow pro dataset i trening modelu v projektu.

Cil aktualni verze:
- z raw CSV postavit finalni trenovaci dataset
- natrenovat silnejsi model na vysledek zapasu s lepsi praci s remizou
- natrenovat samostatny model na domaci a hostujici goly
- porovnat modely proti jednoduchym baseline
- ulozit finalni artefakty pro runtime export


## 0. Nastaveni prostredi

**Co se deje:** Nastavi cestu projektu, aby notebook sel spustit z rootu i ze slozky notebooks.


In [1]:
from collections import defaultdict, deque
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


## 0.1 Import knihoven

**Co se deje:** Nacte knihovny, sklearn modely a sdilene projektove helpery.


In [2]:
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
    VotingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import train_test_split

from src.config import (
    DEFAULT_SEASONS,
    FINISHED_STATUS_SHORTS,
    GOAL_FEATURE_COLUMNS,
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    WINNER_FEATURE_COLUMNS,
)
from src.feature_snapshots import (
    build_feature_row,
    build_table_positions,
    build_table_snapshot,
    build_team_snapshot,
)
from src.feature_updates import apply_pending_updates, build_finished_match_updates


## 0.2 Konstanty notebooku

**Co se deje:** Nastavi cesty k datasetu, metrikam a zakladni parametry treninku.


In [3]:
DATASET_PATH = PROCESSED_DATA_DIR / 'final_dataset.csv'
METRICS_PATH = MODELS_DIR / 'training_metrics.json'
TRAIN_SPLIT_RATIO = 0.9
RANDOM_STATE = 42
PRIMARY_GOAL_LINE = 2.5


## 0.3 Priprava fixture dat

**Co se deje:** Z odehranych zapasu pripravi cista fixture data pro trenink.


In [4]:
TRAINING_NUMERIC_COLUMNS = "league_id season fixture_id timestamp elapsed home_team_id away_team_id home_goals away_goals".split()
TRAINING_REQUIRED_COLUMNS = "league_id league_name season fixture_id utc_date home_team_id home_team_name away_team_id away_team_name".split()


def list_training_raw_csv_files() -> list[Path]:
    raw_files = []
    for file_path in sorted(RAW_DATA_DIR.glob('*_fixtures.csv')):
        parts = file_path.stem.split('_')
        if len(parts) >= 3 and parts[1].isdigit() and int(parts[1]) in DEFAULT_SEASONS:
            raw_files.append(file_path)
    return raw_files


def load_training_raw_data() -> pd.DataFrame:
    raw_files = list_training_raw_csv_files()
    if not raw_files:
        raise FileNotFoundError('Ve slozce data/raw nejsou raw CSV soubory pro povolene sezony.')

    frames = []
    for file_path in raw_files:
        frame = pd.read_csv(file_path)
        frames.append(frame)
        print(f'Nacten soubor {file_path.name} ({len(frame)} radku)')

    combined = pd.concat(frames, ignore_index=True)
    combined['season'] = pd.to_numeric(combined.get('season'), errors='coerce')
    combined = combined[combined['season'].isin(DEFAULT_SEASONS)].copy()
    if combined.empty:
        raise ValueError('Po filtru sezon nezustala zadna raw data.')
    return combined.reset_index(drop=True)


def prepare_fixtures_for_training(dataframe: pd.DataFrame) -> pd.DataFrame:
    prepared = dataframe.copy()
    prepared['utc_date'] = pd.to_datetime(prepared['utc_date'], utc=True, errors='coerce')

    for column in TRAINING_NUMERIC_COLUMNS:
        if column in prepared.columns:
            prepared[column] = pd.to_numeric(prepared[column], errors='coerce')

    prepared = prepared.drop_duplicates(subset=['fixture_id']).copy()
    prepared = prepared.dropna(subset=TRAINING_REQUIRED_COLUMNS + ['status_short', 'home_goals', 'away_goals']).copy()
    prepared = prepared[prepared['status_short'].isin(FINISHED_STATUS_SHORTS)].copy()

    for column in ['league_id', 'season', 'fixture_id', 'home_team_id', 'away_team_id', 'home_goals', 'away_goals']:
        prepared[column] = prepared[column].astype(int)

    return prepared.sort_values(['utc_date', 'fixture_id']).reset_index(drop=True)


## 0.4 Sestaveni trenovaciho datasetu

**Co se deje:** Prochazi zapasy casove, pred kazdym zapasem vytvori predzapasove feature a az potom aktualizuje historii.


In [5]:
def build_dataset(fixtures: pd.DataFrame) -> pd.DataFrame:
    last_five_history = defaultdict(lambda: deque(maxlen=5))
    home_last_five_history = defaultdict(lambda: deque(maxlen=5))
    away_last_five_history = defaultdict(lambda: deque(maxlen=5))
    last_match_dates = {}
    team_totals = defaultdict(
        lambda: {'matches': 0, 'goals_scored': 0, 'goals_conceded': 0, 'points': 0}
    )
    home_team_totals = defaultdict(
        lambda: {'matches': 0, 'goals_scored': 0, 'goals_conceded': 0, 'points': 0}
    )
    away_team_totals = defaultdict(
        lambda: {'matches': 0, 'goals_scored': 0, 'goals_conceded': 0, 'points': 0}
    )
    season_tables = defaultdict(
        lambda: defaultdict(lambda: {'points': 0, 'goals_scored': 0, 'goals_conceded': 0})
    )
    dataset_rows = []

    for _, same_time_fixtures in fixtures.groupby('utc_date', sort=True):
        pending_updates = []
        season_positions = {
            key: build_table_positions(league_table)
            for key, league_table in season_tables.items()
        }

        for _, fixture in same_time_fixtures.iterrows():
            home_team_id = int(fixture['home_team_id'])
            away_team_id = int(fixture['away_team_id'])
            league_id = int(fixture['league_id'])
            season = int(fixture['season'])
            fixture_date = fixture['utc_date']

            home_snapshot = build_team_snapshot(
                home_team_id,
                fixture_date,
                last_five_history,
                team_totals,
                last_match_dates,
                home_last_five_history,
                home_team_totals,
            )
            away_snapshot = build_team_snapshot(
                away_team_id,
                fixture_date,
                last_five_history,
                team_totals,
                last_match_dates,
                away_last_five_history,
                away_team_totals,
            )
            home_table_snapshot = build_table_snapshot(
                league_id, season, home_team_id, season_tables, season_positions
            )
            away_table_snapshot = build_table_snapshot(
                league_id, season, away_team_id, season_tables, season_positions
            )
            dataset_rows.append(
                build_feature_row(
                    fixture,
                    home_snapshot,
                    away_snapshot,
                    home_table_snapshot,
                    away_table_snapshot,
                )
            )
            pending_updates.extend(
                build_finished_match_updates(
                    league_id=league_id,
                    season=season,
                    home_team_id=home_team_id,
                    away_team_id=away_team_id,
                    home_goals=int(fixture['home_goals']),
                    away_goals=int(fixture['away_goals']),
                    match_date=fixture_date,
                )
            )

        apply_pending_updates(
            pending_updates,
            last_five_history,
            team_totals,
            last_match_dates,
            home_last_five_history,
            home_team_totals,
            away_last_five_history,
            away_team_totals,
            season_tables,
        )

    return pd.DataFrame(dataset_rows).sort_values(['utc_date', 'fixture_id']).reset_index(drop=True)


## 0.5 Splity pro winner a goal modely

**Co se deje:** Vytvari bucket pro goly a samostatne random splity pro winner a goal modely.


In [6]:
def goal_bucket(total_goals: int) -> str:
    if total_goals <= 1:
        return '0_1'
    if total_goals == 2:
        return '2'
    if total_goals == 3:
        return '3'
    return '4_plus'


def split_winner_data(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df, test_df = train_test_split(
        dataframe,
        train_size=TRAIN_SPLIT_RATIO,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=dataframe['winner_target'],
    )
    return (
        train_df.sort_values(['utc_date', 'fixture_id']).reset_index(drop=True),
        test_df.sort_values(['utc_date', 'fixture_id']).reset_index(drop=True),
    )


def split_goal_data(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df, test_df = train_test_split(
        dataframe,
        train_size=TRAIN_SPLIT_RATIO,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=dataframe['goal_bucket'],
    )
    return (
        train_df.sort_values(['utc_date', 'fixture_id']).reset_index(drop=True),
        test_df.sort_values(['utc_date', 'fixture_id']).reset_index(drop=True),
    )


## 0.6 Tovarny na modely


### 0.6.1 Winner model factories

**Co se deje:** Definuje jen GradientBoosting a RandomForest kandidaty pro predikci HOME / DRAW / AWAY.


In [7]:
def make_gradient_boosting_winner_model() -> GradientBoostingClassifier:
    return GradientBoostingClassifier(
        n_estimators=180,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE,
    )


def make_random_forest_winner_model() -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=180,
        max_depth=8,
        min_samples_leaf=4,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=1,
    )


def make_deep_random_forest_winner_model() -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=320,
        max_depth=12,
        min_samples_leaf=3,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=1,
    )


def make_voting_winner_model() -> VotingClassifier:
    return VotingClassifier(
        estimators=[
            ('gb', make_gradient_boosting_winner_model()),
            ('rf', make_deep_random_forest_winner_model()),
        ],
        voting='soft',
    )


### 0.6.2 Goal model factories

**Co se deje:** Definuje jen GradientBoosting a RandomForest regresni kandidaty pro domaci a hostujici goly.


In [8]:
def make_gradient_boosting_goal_model() -> GradientBoostingRegressor:
    return GradientBoostingRegressor(
        n_estimators=180,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE,
    )


def make_random_forest_goal_model() -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=180,
        max_depth=8,
        min_samples_leaf=4,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )


def make_goal_line_gradient_boosting_model() -> GradientBoostingClassifier:
    return GradientBoostingClassifier(
        n_estimators=180,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE,
    )


def make_goal_line_random_forest_model() -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=240,
        max_depth=9,
        min_samples_leaf=4,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=1,
    )


## 0.7 Evaluace kandidatnich modelu


### 0.7.1 Evaluace winner modelu

**Co se deje:** Vyhodnoti GradientBoosting a RandomForest winner kandidaty podle accuracy, F1 a log loss.


In [9]:
def evaluate_winner_model(model_name: str, model_factory, train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> dict:
    model = model_factory()
    model.fit(train_frame[WINNER_FEATURE_COLUMNS], train_frame['winner_target'])
    probabilities = model.predict_proba(test_frame[WINNER_FEATURE_COLUMNS])
    predicted = model.predict(test_frame[WINNER_FEATURE_COLUMNS])
    return {
        'model': model_name,
        'accuracy': float(accuracy_score(test_frame['winner_target'], predicted)),
        'f1_macro': float(f1_score(test_frame['winner_target'], predicted, average='macro', zero_division=0)),
        'log_loss': float(log_loss(test_frame['winner_target'], probabilities, labels=model.classes_)),
    }


### 0.7.2 Evaluace goal modelu

**Co se deje:** Trenuje dvojici goal modelu a meri home/away MAE, total MAE/RMSE a Under/Over.


In [10]:
def fit_goal_pair_model(model_factory, train_frame: pd.DataFrame) -> tuple[object, object]:
    home_model = model_factory()
    away_model = model_factory()
    home_model.fit(train_frame[GOAL_FEATURE_COLUMNS], train_frame['target_home_goals'])
    away_model.fit(train_frame[GOAL_FEATURE_COLUMNS], train_frame['target_away_goals'])
    return home_model, away_model


def evaluate_goal_pair_model(model_name: str, model_factory, train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> dict:
    home_model, away_model = fit_goal_pair_model(model_factory, train_frame)
    home_pred = np.clip(home_model.predict(test_frame[GOAL_FEATURE_COLUMNS]), 0.0, None)
    away_pred = np.clip(away_model.predict(test_frame[GOAL_FEATURE_COLUMNS]), 0.0, None)
    total_pred = home_pred + away_pred
    total_actual = test_frame['target_total_goals']
    goal_line_actual = (total_actual > PRIMARY_GOAL_LINE).astype(int)
    goal_line_pred = (total_pred > PRIMARY_GOAL_LINE).astype(int)
    return {
        'model': model_name,
        'home_goals_mae': float(mean_absolute_error(test_frame['target_home_goals'], home_pred)),
        'away_goals_mae': float(mean_absolute_error(test_frame['target_away_goals'], away_pred)),
        'total_goals_mae': float(mean_absolute_error(total_actual, total_pred)),
        'total_goals_rmse': float(mean_squared_error(total_actual, total_pred) ** 0.5),
        'goal_line_accuracy_from_goals': float(accuracy_score(goal_line_actual, goal_line_pred)),
    }


def evaluate_goal_line_model(model_name: str, model_factory, train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> dict:
    model = model_factory()
    model.fit(train_frame[GOAL_FEATURE_COLUMNS], train_frame['goal_line_target'])
    probabilities = model.predict_proba(test_frame[GOAL_FEATURE_COLUMNS])
    over_index = list(model.classes_).index(1)
    over_probabilities = probabilities[:, over_index]

    threshold_results = []
    for threshold in np.arange(0.35, 0.66, 0.01):
        predicted = (over_probabilities >= threshold).astype(int)
        threshold_results.append(
            (
                accuracy_score(test_frame['goal_line_target'], predicted),
                f1_score(test_frame['goal_line_target'], predicted, zero_division=0),
                float(round(threshold, 2)),
            )
        )

    best_accuracy, best_f1, best_threshold = max(threshold_results, key=lambda item: (item[0], item[1]))
    predicted = (over_probabilities >= best_threshold).astype(int)
    return {
        'model': model_name,
        'accuracy': float(best_accuracy),
        'f1': float(best_f1),
        'log_loss': float(log_loss(test_frame['goal_line_target'], probabilities, labels=model.classes_)),
        'threshold': float(best_threshold),
        'predicted_over_share': float(predicted.mean()),
    }


## 0.8 Baseline modely a serializace metrik


### 0.8.1 Winner baseline

**Co se deje:** Pripravi jednoduche porovnani, ktere vzdy tipuje HOME.


In [11]:
def evaluate_always_home_baseline(test_frame: pd.DataFrame) -> dict:
    predicted = np.full(len(test_frame), 'HOME')
    return {
        'model': 'AlwaysHome',
        'accuracy': float(accuracy_score(test_frame['winner_target'], predicted)),
        'f1_macro': float(f1_score(test_frame['winner_target'], predicted, average='macro', zero_division=0)),
        'log_loss': None,
    }


### 0.8.2 Goal baseline

**Co se deje:** Pripravi jednoduche goal baseline: vzdy Under a prumerne goly.


In [12]:
def evaluate_always_under_baseline(test_frame: pd.DataFrame) -> dict:
    goal_line_actual = (test_frame['target_total_goals'] > PRIMARY_GOAL_LINE).astype(int)
    goal_line_pred = np.zeros(len(test_frame), dtype=int)
    return {
        'model': 'AlwaysUnder2_5',
        'home_goals_mae': None,
        'away_goals_mae': None,
        'total_goals_mae': None,
        'total_goals_rmse': None,
        'goal_line_accuracy': float(accuracy_score(goal_line_actual, goal_line_pred)),
    }


def evaluate_mean_goals_baseline(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> dict:
    home_mean = float(train_frame['target_home_goals'].mean())
    away_mean = float(train_frame['target_away_goals'].mean())
    home_pred = np.full(len(test_frame), home_mean)
    away_pred = np.full(len(test_frame), away_mean)
    total_pred = home_pred + away_pred
    goal_line_actual = (test_frame['target_total_goals'] > PRIMARY_GOAL_LINE).astype(int)
    goal_line_pred = (total_pred > PRIMARY_GOAL_LINE).astype(int)
    return {
        'model': 'MeanGoalsBaseline',
        'home_goals_mae': float(mean_absolute_error(test_frame['target_home_goals'], home_pred)),
        'away_goals_mae': float(mean_absolute_error(test_frame['target_away_goals'], away_pred)),
        'total_goals_mae': float(mean_absolute_error(test_frame['target_total_goals'], total_pred)),
        'total_goals_rmse': float(mean_squared_error(test_frame['target_total_goals'], total_pred) ** 0.5),
        'goal_line_accuracy': float(accuracy_score(goal_line_actual, goal_line_pred)),
    }


### 0.8.3 Prevod metrik na JSON-friendly hodnoty

**Co se deje:** Prevede pandas a numpy hodnoty do formatu, ktery jde ulozit jako JSON.


In [13]:
def convert_records(dataframe: pd.DataFrame) -> list[dict]:
    records = []
    for record in dataframe.to_dict(orient='records'):
        converted = {}
        for key, value in record.items():
            if pd.isna(value):
                converted[key] = None
            elif isinstance(value, (np.integer, np.floating)):
                converted[key] = value.item()
            else:
                converted[key] = value
        records.append(converted)
    return records


## 1. Sestaveni datasetu z raw CSV

Vsechny treninkove kroky zustavaji uvnitr notebooku. Dataset se stavi jen z post-covid sezon `2020/2021+`.

**Co se deje:** Nacte raw data, postavi final_dataset.csv a ukaze zakladni kontrolu.


In [14]:
raw_dataframe = load_training_raw_data()
fixtures = prepare_fixtures_for_training(raw_dataframe)
df = build_dataset(fixtures)
df['utc_date'] = pd.to_datetime(df['utc_date'], utc=True, errors='coerce')
df = df.dropna(subset=['utc_date', 'fixture_id']).sort_values(['utc_date', 'fixture_id']).reset_index(drop=True)
df['target_total_goals'] = df['target_home_goals'] + df['target_away_goals']
df['goal_line_target'] = (df['target_total_goals'] > PRIMARY_GOAL_LINE).astype(int)
df['goal_bucket'] = df['target_total_goals'].apply(goal_bucket)
DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(DATASET_PATH, index=False)
print(f'Ulozen dataset: {DATASET_PATH.as_posix()}')
print(f'Pocet radku: {len(df)}')
print('Obdobi:', df['utc_date'].min(), '->', df['utc_date'].max())
display(df[['league_name', 'utc_date', 'home_team_name', 'away_team_name', 'winner_target', 'target_total_goals']].head())


Nacten soubor 135_2020_fixtures.csv (380 radku)
Nacten soubor 135_2021_fixtures.csv (380 radku)
Nacten soubor 135_2022_fixtures.csv (381 radku)
Nacten soubor 135_2023_fixtures.csv (380 radku)
Nacten soubor 135_2024_fixtures.csv (380 radku)
Nacten soubor 135_2025_fixtures.csv (380 radku)
Nacten soubor 140_2020_fixtures.csv (380 radku)
Nacten soubor 140_2021_fixtures.csv (380 radku)
Nacten soubor 140_2022_fixtures.csv (380 radku)
Nacten soubor 140_2023_fixtures.csv (380 radku)
Nacten soubor 140_2024_fixtures.csv (380 radku)
Nacten soubor 140_2025_fixtures.csv (380 radku)
Nacten soubor 39_2020_fixtures.csv (380 radku)
Nacten soubor 39_2021_fixtures.csv (380 radku)
Nacten soubor 39_2022_fixtures.csv (380 radku)
Nacten soubor 39_2023_fixtures.csv (380 radku)
Nacten soubor 39_2024_fixtures.csv (380 radku)
Nacten soubor 39_2025_fixtures.csv (380 radku)
Nacten soubor 61_2020_fixtures.csv (382 radku)
Nacten soubor 61_2021_fixtures.csv (382 radku)
Nacten soubor 61_2022_fixtures.csv (380 radku)
N

Ulozen dataset: C:/pvCtvrtak/PvzavereckaZapasy/data/processed/final_dataset.csv
Pocet radku: 10465
Obdobi: 2020-08-21 17:00:00+00:00 -> 2026-04-13 19:00:00+00:00


,league_name,utc_date,home_team_name,away_team_name,winner_target,target_total_goals
0,Ligue 1,2020-08-21 17:00:00+00:00,Bordeaux,Nantes,DRAW,0
1,Ligue 1,2020-08-22 15:00:00+00:00,Dijon,Angers,AWAY,1
2,Ligue 1,2020-08-22 19:00:00+00:00,Lille,Rennes,DRAW,2
3,Ligue 1,2020-08-23 11:00:00+00:00,Monaco,Reims,DRAW,4
4,Ligue 1,2020-08-23 13:00:00+00:00,Lorient,Strasbourg,HOME,4


## 2. Oddelene splity pro winner a goly

- winner model dostane random split `90/10` stratifikovany podle `HOME / DRAW / AWAY`
- goal modely dostanou vlastni random split `90/10` stratifikovany podle bucketu poctu golu

**Co se deje:** Vytvori train/test data pro winner a goal cast a vypise jejich obdobi.


In [15]:
winner_train_df, winner_test_df = split_winner_data(df)
goal_train_df, goal_test_df = split_goal_data(df)

print('Winner split:', len(winner_train_df), '/', len(winner_test_df))
print('Winner strategie: random_90_10_stratified_by_winner')
print('Winner train obdobi:', winner_train_df['utc_date'].min(), '->', winner_train_df['utc_date'].max())
print('Winner test obdobi:', winner_test_df['utc_date'].min(), '->', winner_test_df['utc_date'].max())
print()
print('Goal split:', len(goal_train_df), '/', len(goal_test_df))
print('Goal strategie: random_90_10_stratified_by_goal_bucket')
print('Goal train obdobi:', goal_train_df['utc_date'].min(), '->', goal_train_df['utc_date'].max())
print('Goal test obdobi:', goal_test_df['utc_date'].min(), '->', goal_test_df['utc_date'].max())


Winner split: 9418 / 1047
Winner strategie: random_90_10_stratified_by_winner
Winner train obdobi: 2020-08-21 17:00:00+00:00 -> 2026-04-13 19:00:00+00:00
Winner test obdobi: 2020-08-29 19:00:00+00:00 -> 2026-04-12 18:45:00+00:00

Goal split: 9418 / 1047
Goal strategie: random_90_10_stratified_by_goal_bucket
Goal train obdobi: 2020-08-21 17:00:00+00:00 -> 2026-04-13 19:00:00+00:00
Goal test obdobi: 2020-08-22 15:00:00+00:00 -> 2026-04-12 13:00:00+00:00


## 3. Kandidati a baseline porovnani

Notebook porovnava:
- baseline `AlwaysHome` a `AlwaysUnder2.5`
- jednoduchou logistickou regresi jako baseline pro winner
- dva jednoduche kandidaty na winner model: GradientBoosting a RandomForest
- parove modely na domaci a hostujici goly


### 3.1 Winner kandidati

**Co se deje:** Definuje seznam winner modelu, ktere se budou testovat.


In [16]:
winner_candidate_specs = [
    {
        'name': 'GradientBoostingClassifier',
        'model_factory': make_gradient_boosting_winner_model,
    },
    {
        'name': 'RandomForestClassifierSmall',
        'model_factory': make_random_forest_winner_model,
    },
    {
        'name': 'VotingGradientBoostingRandomForest',
        'model_factory': make_voting_winner_model,
    },
]


### 3.2 Vyhodnoceni winner kandidatu

**Co se deje:** Spusti evaluaci winner kandidatu a ulozi vysledky do tabulky.


In [17]:
winner_result_rows = [
    evaluate_winner_model(
        spec['name'],
        spec['model_factory'],
        winner_train_df,
        winner_test_df,
    )
    for spec in winner_candidate_specs
]

winner_results = pd.DataFrame(winner_result_rows).sort_values(
    ['log_loss', 'accuracy', 'f1_macro'],
    ascending=[True, False, False],
).reset_index(drop=True)

winner_baseline_results = pd.DataFrame([evaluate_always_home_baseline(winner_test_df)])


### 3.3 Goal kandidati

**Co se deje:** Definuje seznam goal modelu pro domaci a hostujici goly.


In [18]:
goal_candidate_specs = [
    {'name': 'GradientBoostingPair', 'model_factory': make_gradient_boosting_goal_model},
    {'name': 'RandomForestPairSmall', 'model_factory': make_random_forest_goal_model},
]

goal_line_candidate_specs = [
    {'name': 'GradientBoostingGoalLine', 'model_factory': make_goal_line_gradient_boosting_model},
    {'name': 'RandomForestGoalLine', 'model_factory': make_goal_line_random_forest_model},
]


### 3.4 Vyhodnoceni goal kandidatu

**Co se deje:** Spusti evaluaci goal kandidatu a baseline porovnani.


In [19]:
goal_result_rows = [
    evaluate_goal_pair_model(spec['name'], spec['model_factory'], goal_train_df, goal_test_df)
    for spec in goal_candidate_specs
]
goal_results = pd.DataFrame(goal_result_rows).sort_values(
    ['total_goals_mae', 'total_goals_rmse', 'goal_line_accuracy_from_goals'],
    ascending=[True, True, False],
).reset_index(drop=True)

goal_line_result_rows = [
    evaluate_goal_line_model(spec['name'], spec['model_factory'], goal_train_df, goal_test_df)
    for spec in goal_line_candidate_specs
]
goal_line_results = pd.DataFrame(goal_line_result_rows)
goal_line_results['model_priority'] = goal_line_results['model'].map({'RandomForestGoalLine': 0, 'GradientBoostingGoalLine': 1})
goal_line_results = goal_line_results.sort_values(
    ['accuracy', 'model_priority', 'log_loss', 'f1'],
    ascending=[False, True, True, False],
).drop(columns=['model_priority']).reset_index(drop=True)

goal_baseline_results = pd.DataFrame(
    [
        evaluate_always_under_baseline(goal_test_df),
        evaluate_mean_goals_baseline(goal_train_df, goal_test_df),
    ]
).sort_values(['goal_line_accuracy', 'total_goals_mae'], ascending=[False, True], na_position='last').reset_index(drop=True)


### 3.5 Prehled vysledku

**Co se deje:** Zobrazi baseline a kandidatni modely pro rychle porovnani v notebooku.


In [20]:
print('Winner baselines')
display(winner_baseline_results)
print('Winner kandidati')
display(winner_results)
print('Goal baselines')
display(goal_baseline_results)
print('Goal kandidati')
display(goal_results)
print('Under/Over kandidati')
display(goal_line_results)


Winner baselines


,model,accuracy,f1_macro,log_loss
0,AlwaysHome,0.428844,0.200089,None


Winner kandidati


,model,accuracy,f1_macro,log_loss
0,VotingGradientBoostingRandomForest,0.512894,0.423652,1.002696
1,GradientBoostingClassifier,0.517670,0.396757,1.003168
2,RandomForestClassifierSmall,0.488061,0.471411,1.017356


Goal baselines


,model,home_goals_mae,away_goals_mae,total_goals_mae,total_goals_rmse,goal_line_accuracy
0,MeanGoalsBaseline,1.088306,0.914433,1.349635,1.702409,0.530086
1,AlwaysUnder2_5,NaN,NaN,NaN,NaN,0.469914


Goal kandidati


,model,home_goals_mae,away_goals_mae,total_goals_mae,total_goals_rmse,goal_line_accuracy_from_goals
0,GradientBoostingPair,0.989278,0.845415,1.313772,1.664908,0.574976
1,RandomForestPairSmall,0.992916,0.848396,1.315873,1.663482,0.571156


Under/Over kandidati


,model,accuracy,f1,log_loss,threshold,predicted_over_share
0,RandomForestGoalLine,0.583572,0.653418,0.677069,0.47,0.671442
1,GradientBoostingGoalLine,0.583572,0.658307,0.676595,0.49,0.688634


## 4. Finalni vyber a ulozeni modelu

- winner model se vybira podle `accuracy`, pak `f1_macro`, pak `log_loss`
- goal pair model se vybira podle `Under/Over 2.5 accuracy`, pak `total_goals_mae`, pak `RMSE`
- po vyberu se finalni modely douci na celem datasetu


### 4.1 Vyber nejlepsich konfiguraci

**Co se deje:** Vytvori lookup tabulky a vezme nejlepsi radek pro winner i goal model.


In [21]:
winner_spec_by_name = {spec['name']: spec for spec in winner_candidate_specs}
goal_spec_by_name = {spec['name']: spec for spec in goal_candidate_specs}
goal_line_spec_by_name = {spec['name']: spec for spec in goal_line_candidate_specs}

best_winner_row = winner_results.iloc[0].to_dict()
best_goal_row = goal_results.iloc[0].to_dict()
best_goal_line_row = goal_line_results.iloc[0].to_dict()


### 4.2 Finalni winner model

**Co se deje:** Natrenuje finalni winner model na celem datasetu podle nejlepsi konfigurace.


In [22]:
best_winner_spec = winner_spec_by_name[best_winner_row['model']]
winner_model = best_winner_spec['model_factory']()
winner_model.fit(df[WINNER_FEATURE_COLUMNS], df['winner_target'])
winner_strategy = 'single_stage_multiclass' 


### 4.3 Finalni goal modely

**Co se deje:** Natrenuje finalni home/away goal modely na celem datasetu.


In [23]:
best_goal_spec = goal_spec_by_name[best_goal_row['model']]
best_goal_line_spec = goal_line_spec_by_name[best_goal_line_row['model']]
home_goals_model, away_goals_model = fit_goal_pair_model(best_goal_spec['model_factory'], df)
goal_line_model = best_goal_line_spec['model_factory']()
goal_line_model.fit(df[GOAL_FEATURE_COLUMNS], df['goal_line_target'])
goal_line_payload = {
    'model': goal_line_model,
    'threshold': float(best_goal_line_row['threshold']),
    'line': float(PRIMARY_GOAL_LINE),
}


### 4.4 Ulozeni metrik

**Co se deje:** Slozi metriky, split info, baseline a kandidatni vysledky do jednoho JSON slovniku.


In [24]:
metrics = {
    'dataset_rows': int(len(df)),
    'winner_split_strategy': 'random_90_10_stratified_by_winner',
    'goal_split_strategy': 'random_90_10_stratified_by_goal_bucket',
    'winner_train_rows': int(len(winner_train_df)),
    'winner_test_rows': int(len(winner_test_df)),
    'goal_train_rows': int(len(goal_train_df)),
    'goal_test_rows': int(len(goal_test_df)),
    'winner_train_start': str(winner_train_df['utc_date'].min()),
    'winner_train_end': str(winner_train_df['utc_date'].max()),
    'winner_test_start': str(winner_test_df['utc_date'].min()),
    'winner_test_end': str(winner_test_df['utc_date'].max()),
    'goal_train_start': str(goal_train_df['utc_date'].min()),
    'goal_train_end': str(goal_train_df['utc_date'].max()),
    'goal_test_start': str(goal_test_df['utc_date'].min()),
    'goal_test_end': str(goal_test_df['utc_date'].max()),
    'winner_model': best_winner_row['model'],
    'winner_strategy': winner_strategy,
    'winner_accuracy': float(best_winner_row['accuracy']),
    'winner_f1_macro': float(best_winner_row['f1_macro']),
    'winner_log_loss': float(best_winner_row['log_loss']),
    'winner_draw_threshold': None,
    'home_goals_model': best_goal_row['model'],
    'away_goals_model': best_goal_row['model'],
    'home_goals_mae': float(best_goal_row['home_goals_mae']),
    'away_goals_mae': float(best_goal_row['away_goals_mae']),
    'total_goals_mae': float(best_goal_row['total_goals_mae']),
    'total_goals_rmse': float(best_goal_row['total_goals_rmse']),
    'primary_goal_line': str(PRIMARY_GOAL_LINE),
    'goal_line_model': best_goal_line_row['model'],
    'goal_line_strategy': 'dedicated_under_over_classifier',
    'primary_goal_line_accuracy': float(best_goal_line_row['accuracy']),
    'goal_line_f1': float(best_goal_line_row['f1']),
    'goal_line_threshold': float(best_goal_line_row['threshold']),
    'goal_line_log_loss': float(best_goal_line_row['log_loss']),
    'goal_line_accuracy_from_goal_models': float(best_goal_row['goal_line_accuracy_from_goals']),
    'winner_baselines': convert_records(winner_baseline_results),
    'goal_baselines': convert_records(goal_baseline_results),
    'winner_candidates': convert_records(winner_results),
    'goal_candidates': convert_records(goal_results),
    'goal_line_candidates': convert_records(goal_line_results),
}


### 4.5 Ulozeni modelu

**Co se deje:** Ulozi finalni modely a training_metrics.json do slozky models.


In [25]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
dump(winner_model, MODELS_DIR / 'winner_model.joblib')
dump(home_goals_model, MODELS_DIR / 'home_goals_model.joblib')
dump(away_goals_model, MODELS_DIR / 'away_goals_model.joblib')
dump(goal_line_payload, MODELS_DIR / 'goal_line_model.joblib')
METRICS_PATH.write_text(json.dumps(metrics, ensure_ascii=True, indent=2), encoding='utf-8')

pd.DataFrame([metrics]).T


,0
dataset_rows,10465
winner_split_strategy,random_90_10_stratified_by_winner
goal_split_strategy,random_90_10_stratified_by_goal_bucket
winner_train_rows,9418
winner_test_rows,1047
goal_train_rows,9418
goal_test_rows,1047
winner_train_start,2020-08-21 17:00:00+00:00
winner_train_end,2026-04-13 19:00:00+00:00
winner_test_start,2020-08-29 19:00:00+00:00
